# Indian Sign Language (ISL) Recognition — GPU Model Training (Google Colab / Kaggle)

This notebook trains the **Research Paper CNN Architecture** (30,155,955 parameters from IEEE RAIT 2025 paper `2507.20414v2`) or the **Compact CNN** using GPU acceleration (Tesla T4 / V100 / A100).

### Instructions:
1. Go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Run all cells sequentially.
3. At the end, download `best_model.keras` and place it in your local `artifacts/` folder.

In [ ]:
# 1. Install dependencies
!pip install kagglehub opencv-python-headless pandas scikit-learn matplotlib tqdm pyyaml

In [ ]:
# 2. Download Dataset via KaggleHub
import kagglehub
from pathlib import Path

print("Downloading Indian Sign Language Dataset...")
raw_path = kagglehub.dataset_download("prathumarikeri/indian-sign-language-isl")
dataset_root = Path(raw_path)
if (dataset_root / "Indian").exists():
    dataset_root = dataset_root / "Indian"
print("Dataset root located at:", dataset_root)
print("Total class folders:", len(list(dataset_root.iterdir())))

In [ ]:
# 3. Unified Preprocessing Pipeline (Identical to local src/preprocessing.py)
import cv2
import numpy as np

def preprocess_image(image_input, target_size=(64, 64), threshold_value=90, use_canny=True, canny_low=50, canny_high=150, normalize=True):
    if isinstance(image_input, (str, Path)):
        img = cv2.imread(str(image_input))
        if img is None:
            raise ValueError(f"Could not read image: {image_input}")
    else:
        img = image_input.copy()
        
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img
        
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    if use_canny:
        processed = cv2.Canny(blurred, canny_low, canny_high)
    else:
        _, processed = cv2.threshold(blurred, threshold_value, 255, cv2.THRESH_BINARY)
        
    if (processed.shape[1], processed.shape[0]) != target_size:
        processed = cv2.resize(processed, target_size, interpolation=cv2.INTER_AREA)
        
    if normalize:
        processed_arr = processed.astype(np.float32) / 255.0
    else:
        processed_arr = processed
        
    if len(processed_arr.shape) == 2:
        processed_arr = np.expand_dims(processed_arr, axis=-1)
    return processed_arr

In [ ]:
# 4. Stratified Dataset Splitting (80/10/10)
import pandas as pd
from sklearn.model_selection import train_test_split

classes = sorted([d.name for d in dataset_root.iterdir() if d.is_dir()])
class_to_idx = {name: idx for idx, name in enumerate(classes)}

records = []
for c_name, c_idx in class_to_idx.items():
    for f in (dataset_root / c_name).iterdir():
        if f.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            records.append({"path": str(f), "class_name": c_name, "class_idx": c_idx})

df = pd.DataFrame(records)
print(f"Total images: {len(df)} across {len(classes)} classes")

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["class_idx"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["class_idx"], random_state=42)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# 5. Fast Batch Generator with Safe Augmentation
import random
import tensorflow as tf

def apply_safe_augmentation(img):
    h, w = img.shape[:2]
    angle = random.uniform(-10, 10)
    scale = random.uniform(0.95, 1.05)
    tx, ty = random.uniform(-3, 3), random.uniform(-3, 3)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    M[0, 2] += tx; M[1, 2] += ty
    aug = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    if len(aug.shape) == 2:
        aug = np.expand_dims(aug, axis=-1)
    return aug

class ISLColabGenerator:
    def __init__(self, df, batch_size=32, target_size=(64, 64), augment=False):
        self.df = df
        self.batch_size = batch_size
        self.target_size = target_size
        self.augment = augment
        self.indices = list(range(len(self.df)))
        
    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))
        
    def flow(self):
        while True:
            random.shuffle(self.indices)
            for i in range(len(self)):
                batch_idx = self.indices[i * self.batch_size : (i + 1) * self.batch_size]
                X, y = [], []
                for b in batch_idx:
                    row = self.df.iloc[b]
                    im = preprocess_image(row["path"], target_size=self.target_size)
                    if self.augment:
                        im = apply_safe_augmentation(im)
                    X.append(im)
                    y.append(row["class_idx"])
                yield np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

train_gen = ISLColabGenerator(train_df, batch_size=32, target_size=(64, 64), augment=True)
val_gen = ISLColabGenerator(val_df, batch_size=32, target_size=(64, 64), augment=False)

In [ ]:
# 6. Define Research Paper CNN Architecture (Table 1: 30,155,955 parameters)
from tensorflow.keras import layers, models, optimizers

def build_paper_cnn(input_shape=(64, 64, 1), num_classes=35):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(24, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.Dropout(0.25),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.Dropout(0.25),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.Dropout(0.25),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.Dropout(0.25),
        
        layers.Flatten(),
        layers.Dense(2352, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ], name='ISL_Paper_CNN')
    
    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_paper_cnn((64, 64, 1), len(classes))
model.summary()

In [ ]:
# 7. Train on GPU with ModelCheckpoint & EarlyStopping
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

callbacks = [
    ModelCheckpoint('best_model.keras', monitor='val_loss', mode='min', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

history = model.fit(
    train_gen.flow(),
    steps_per_epoch=len(train_gen),
    epochs=25,
    validation_data=val_gen.flow(),
    validation_steps=len(val_gen),
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# 8. Evaluate on Held-Out Test Set
from sklearn.metrics import classification_report, accuracy_score

print("Evaluating on held-out test set...")
X_test = np.array([preprocess_image(p, (64, 64)) for p in test_df['path']], dtype=np.float32)
y_test = test_df['class_idx'].values

preds = model.predict(X_test, batch_size=32)
y_pred = np.argmax(preds, axis=1)

acc = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {acc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=classes))

In [ ]:
# 9. Download Trained Model to Local Machine
from google.colab import files
files.download('best_model.keras')